## 1.加载数据
在上一节 3.TextCNN中，我们已经存储了预处理好的数据，所以，我们这里只需要加载

另外，由于RNN处理变动长度序列，所以我们额外读取序列程度sqlen

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

#使用3.TextCNN保存的数据
train_load = torch.load("data/Malo/Malo_Train.pth", weights_only=True)
# 使用 TensorDataset 恢复数据集
train_data = TensorDataset(train_load["data"],train_load["lengths"], train_load["labels"])

valid_load = torch.load("data/Malo/Malo_Valid.pth", weights_only=True)
# 使用 TensorDataset 恢复数据集
valid_data = TensorDataset(valid_load["data"],valid_load["lengths"], valid_load["labels"])

test_load = torch.load("data/Malo/Malo_Test.pth", weights_only=True)
# 使用 TensorDataset 恢复数据集
test_data = TensorDataset(test_load["data"],test_load["lengths"], test_load["labels"])

In [ ]:
batch_size = 50

train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(valid_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)

## 2.定义模型

### 2.1 LSTM模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


#LSTM模型
class TextLSTM(nn.Module):
    def __init__(self, embed_model, vocab_size, embedding_dim, hidden_dim, output_size, embed_freeze=True):
        """
        初始化 TextLSTM 模型的各个组件。
        
        参数:
        - vocab_size: 词汇表大小，表示模型可以接受的词的总数量。
        - embedding_dim: 嵌入维度大小，每个词嵌入为 embedding_dim 维向量。
        - hidden_dim: LSTM 的隐藏状态维度。
        - output_size: 输出类别数量（如 3 表示三分类）。
        """
        super(TextLSTM, self).__init__()

        # 1.嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 载入预训练的权重
        self.embedding.weight = nn.Parameter(torch.from_numpy(embed_model.vectors)) # all vectors
        # 冻结嵌入层的训练，设定为false则一起训练
        if embed_freeze:
            self.embedding.requires_grad = False
        
        # LSTM层: 用于处理序列数据的循环神经网络
        self.lstm = nn.LSTM(
            input_size=embedding_dim,   # 输入维度
            hidden_size=hidden_dim,     # 隐藏层维度
            batch_first=True            # 输入和输出的第一个维度是 batch_size
        )

        # 全连接层: 将 LSTM 的最后隐藏状态映射到类别分数
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x, lengths):
        """
        定义前向传播逻辑。
        
        参数:
        - x: 输入张量，形状为 (batch_size, seq_length)
        - lengths: 每个序列的实际长度列表，形状为 (batch_size,)
        
        返回:
        - out: 分类得分，形状为 (batch_size, output_size)
        """
        # 1. 嵌入层: 将词索引转为嵌入向量
        # 输入: x, 形状 (batch_size, seq_length)
        # 输出: x, 形状 (batch_size, seq_length, embedding_dim)
        x = self.embedding(x)

        # 2. 使用 Packed Sequence
        # 将填充序列转换为 PackedSequence
        packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        # 3. LSTM层: 处理序列数据
        # 输入: packed_x
        # 输出:
        # - packed_out: PackedSequence 对象
        # - (hidden, cell): LSTM 的最后隐藏状态
        packed_out, (hidden, cell) = self.lstm(packed_x)

        # 4. 解包序列（如果需要整个序列输出，可以解包）
        # lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True)

        # 5. 获取最后一层的隐藏状态
        # hidden[-1] 表示最后一层的隐藏状态 (batch_size, hidden_dim)
        final_hidden = hidden[-1]

        # 6. 全连接层: 将隐藏状态映射到类别分数
        # 输入: final_hidden, 形状 (batch_size, hidden_dim)
        # 输出: out, 形状 (batch_size, output_size)
        out = self.fc(final_hidden)

        return F.log_softmax(out, dim=1)

### 2.2 Simple RNN 模型

In [ ]:
#simple RNN 模型
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class TextRNN(nn.Module):
    def __init__(self, embed_model, vocab_size, embedding_dim, hidden_dim, output_size, embed_freeze=True):
        """
        初始化 TextRNN 模型的各个组件。
        
        参数:
        - vocab_size: 词汇表大小，表示模型可以接受的词的总数量。
        - embedding_dim: 嵌入维度大小，每个词嵌入为 embedding_dim 维向量。
        - hidden_dim: RNN 的隐藏状态维度。
        - output_size: 输出类别数量（如 3 表示三分类）。
        """
        super(TextRNN, self).__init__()

        # 1.嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 载入预训练的权重
        self.embedding.weight = nn.Parameter(torch.from_numpy(embed_model.vectors)) # all vectors
        # 冻结嵌入层的训练，设定为false则一起训练
        if embed_freeze:
            self.embedding.requires_grad = False
        
        # RNN层: 用于处理序列数据的简单循环神经网络
        self.rnn = nn.RNN(
            input_size=embedding_dim,  # 输入维度
            hidden_size=hidden_dim,    # 隐藏层维度
            batch_first=True           # 输入和输出的第一个维度是 batch_size
        )

        # 全连接层: 将 RNN 的最后隐藏状态映射到类别分数
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x, lengths):
        """
        定义前向传播逻辑。
        
        参数:
        - x: 输入张量，形状为 (batch_size, seq_length)
        - lengths: 每个序列的实际长度列表，形状为 (batch_size,)
        
        返回:
        - out: 分类得分，形状为 (batch_size, output_size)
        """
        # 1. 嵌入层: 将词索引转为嵌入向量
        # 输入: x, 形状 (batch_size, seq_length)
        # 输出: x, 形状 (batch_size, seq_length, embedding_dim)
        x = self.embedding(x)

        # 2. 使用 Packed Sequence
        # 将填充序列转换为 PackedSequence
        packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        # 3. RNN层: 处理序列数据
        # 输入: packed_x
        # 输出:
        # - packed_out: PackedSequence 对象
        # - hidden: 形状 (num_layers, batch_size, hidden_dim)
        packed_out, hidden = self.rnn(packed_x)

        # 4. 解包序列（如果需要完整的序列输出）
        # rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True)

        # 5. 获取最后一层的隐藏状态:
        # hidden[-1] 表示最后一层的隐藏状态 (batch_size, hidden_dim)
        final_hidden = hidden[-1]

        # 6. 全连接层: 将隐藏状态映射到类别分数
        # 输入: final_hidden, 形状 (batch_size, hidden_dim)
        # 输出: out, 形状 (batch_size, output_size)
        out = self.fc(final_hidden)

        return F.log_softmax(out, dim=1)

### 2.3 GRU
请注意，这个GRU版本，支持双向，请注意它做了哪些改动

同时，GRU版本加入了多层与dropout，可以尝试给RNN和LSTM加入相似的部分

In [ ]:
class TextGRU(nn.Module):
    def __init__(self, embed_model, vocab_size, embedding_dim, hidden_dim, output_size, num_layers=1, bidirectional=False, dropout=0.5,embed_freeze=True):
        """
        初始化 TextGRU 模型的各个组件。
        
        参数:
        - vocab_size: 词汇表大小。
        - embedding_dim: 嵌入维度大小。
        - hidden_dim: GRU 的隐藏状态维度。
        - output_size: 输出类别数量（如 3 表示三分类）。
        - num_layers: GRU 的层数。
        - bidirectional: 是否使用双向 GRU。
        - dropout: Dropout 概率。
        """
        super(TextGRU, self).__init__()

        # 1.嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 载入预训练的权重
        self.embedding.weight = nn.Parameter(torch.from_numpy(embed_model.vectors)) # all vectors
        # 冻结嵌入层的训练，设定为false则一起训练
        if embed_freeze:
            self.embedding.requires_grad = False
        
        # GRU层: 用于处理序列数据
        self.gru = nn.GRU(
            input_size=embedding_dim,   # 输入维度
            hidden_size=hidden_dim,     # 隐藏层维度
            num_layers=num_layers,      # GRU 层数
            batch_first=True,           # 输入和输出的第一个维度是 batch_size
            bidirectional=bidirectional, # 是否双向
            dropout=dropout if num_layers > 1 else 0  # 多层 GRU 才使用 Dropout
        )

        # Dropout层: 防止过拟合
        self.dropout = nn.Dropout(dropout)

        # 全连接层: 将 GRU 的隐藏状态映射到类别分数
        direction_factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * direction_factor, output_size)

    def forward(self, x, lengths):
        """
        定义前向传播逻辑。
        
        参数:
        - x: 输入张量，形状为 (batch_size, seq_length)
        - lengths: 每个序列的实际长度列表，形状为 (batch_size,)
        
        返回:
        - out: 分类得分，形状为 (batch_size, output_size)
        """
        # 1. 嵌入层: 将词索引转为嵌入向量
        x = self.embedding(x)  # (batch_size, seq_length) -> (batch_size, seq_length, embedding_dim)

        # 2. 使用 PackedSequence
        # 将填充序列转换为 PackedSequence
        packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

        # 3. GRU层: 处理序列数据
        # 输入: packed_x
        # 输出:
        # - packed_out: PackedSequence 对象
        # - hidden: 形状 (num_layers * direction_factor, batch_size, hidden_dim)
        packed_out, hidden = self.gru(packed_x)

        # 4. 解包序列（如果需要完整的序列输出）
        # gru_out, _ = pad_packed_sequence(packed_out, batch_first=True)

        # 5. 获取最后一层的隐藏状态:
        # 对多层 GRU 的最后一层处理
        if self.gru.bidirectional:
            # 拼接正向和反向的隐藏状态
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (batch_size, hidden_dim * 2)
        else:
            # 取最后一层的隐藏状态
            final_hidden = hidden[-1]  # (batch_size, hidden_dim)

        # 6. Dropout层
        final_hidden = self.dropout(final_hidden)

        # 7. 全连接层: 将隐藏状态映射到类别分数
        out = self.fc(final_hidden)  # (batch_size, output_size)

        return F.log_softmax(out, dim=1)

## 3.定义训练与测试函数

In [ ]:
#训练函数与测试函数仍然可以复用（why？因为依然是分类问题）
#定义一个训练函数
def train(args, model, device, train_loader, optimizer, epoch):
    """
    训练函数，支持 PackedSequence
    """
    model.train()  # 将模型切换为训练模式
    for batch_idx, (data, lengths, target) in enumerate(train_loader):  # 加载数据，同时获取序列长度
        data, lengths, target = data.to(device), lengths.to(device), target.to(device)  # 将数据转移到设备
        optimizer.zero_grad()  # 清空梯度缓存
        output = model(data, lengths)  # 传递数据和序列长度
        loss = F.nll_loss(output, target)  # 计算损失
        loss.backward()  # 反向传播
        optimizer.step()  # 更新模型参数


def test(model, device, test_loader):
    """
    测试函数，支持 PackedSequence
    """
    model.eval()  # 将模型切换为评估模式
    test_loss = 0
    correct = 0
    with torch.no_grad():  # 禁用自动求导功能
        for data, lengths, target in test_loader:  # 加载数据，同时获取序列长度
            data, lengths, target = data.to(device), lengths.to(device), target.to(device)  # 转移数据到设备
            output = model(data, lengths)  # 传递数据和序列长度
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # 计算损失并累加
            pred = output.argmax(dim=1, keepdim=True)  # 获取预测值
            correct += pred.eq(target.view_as(pred)).sum().item()  # 累计正确预测数

    test_loss /= len(test_loader.dataset)  # 平均损失
    print("=" * 50)
    print("Evaluation Results:")
    print(f" - Average Loss: {test_loss:.4f}")
    print(f" - Accuracy: {correct}/{len(test_loader.dataset)} "
          f"({100. * correct / len(test_loader.dataset):.2f}%)")
    print("=" * 50)

    return 100. * correct / len(test_loader.dataset)

    


In [ ]:
epochs = 6  #定义最大训练周期
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  #定义设备,CPU或者GPU（Cuda）
print(device)

## 4.加载预训练word2vec

In [ ]:
# 使用gensim模块来加载word2vec model
from gensim.models import KeyedVectors
# 加载模型，一定加载SLIM模型，加载完整版本，内存/显存占据过大
embed_lookup = KeyedVectors.load_word2vec_format('model/GoogleNews-vectors-negative300-SLIM.bin', 
                                                 binary=True)

pretrained_words = []
for word in embed_lookup.key_to_index:
    pretrained_words.append(word)

## 5.定义模型

In [ ]:

vocab_size = len(pretrained_words)
output_size = 3 
embedding_dim = len(embed_lookup[pretrained_words[0]]) # 300-dim vectors

hidden_dim = 100
output_size = 3

# Simple RNN
model_RNN = TextRNN(embed_lookup, vocab_size, embedding_dim, hidden_dim, output_size).to(device)
print(model_RNN)

# LSTM
model_LSTM = TextLSTM(embed_lookup, vocab_size, embedding_dim, hidden_dim, output_size).to(device)
print(model_LSTM)

# GRU
model_GRU = TextGRU(embed_lookup, vocab_size, embedding_dim, hidden_dim, output_size).to(device)  #, num_layers=1, bidirectional=False, dropout=0.5,embed_freeze=True
print(model_GRU)

# GRU_bi 双向GRU模型
model_GRU_bi = TextGRU(embed_lookup, vocab_size, embedding_dim, hidden_dim, output_size, bidirectional=True).to(device)  #, num_layers=1, bidirectional=False, dropout=0.5,embed_freeze=True
print(model_GRU_bi)

# GRU_bi_3 双向GRU，深度为3
model_GRU_bi_3 = TextGRU(embed_lookup, vocab_size, embedding_dim, hidden_dim, output_size, bidirectional=True, num_layers=3).to(device)  #, num_layers=1, bidirectional=False, dropout=0.5,embed_freeze=True
print(model_GRU_bi_3)

## 6.Action

In [ ]:
epochs = 6

In [ ]:
import time
import torch.optim as optim

models_l = [model_RNN,model_LSTM,model_GRU,model_GRU_bi,model_GRU_bi_3]
model_names = ['RNN','LSTM','GRU','GRU_bi','GRU_bi_3']
resl = []
for li in range(len(models_l)):
    model = models_l[li]
    model_name = model_names[li]

    

    optimizer = optim.Adadelta(model.parameters())
    
    for epoch in range(1, epochs + 1):
        tline = {}
        tline['model_name'] = model_name
        tline['epoch']=epoch
        
        start_time = time.time()  #记录开始时间
        train([], model, device, train_loader, optimizer, epoch)  #训练
        train_time = time.time()
        tline['valid_precision'] = test(model, device, valid_loader)  #测试
        test_time = time.time()
        train_duration = train_time - start_time
        test_duration = test_time - train_time

        tline['train_duration']=train_duration
        tline['test_duration']=test_duration
        
        print(f"Epoch {epoch}/{epochs} - Trained in {train_duration:.2f} seconds  - Test in {test_duration:.2f} seconds.\n")

        resl.append(tline)


In [ ]:
import pandas as pd
res_df = pd.DataFrame(resl)

In [ ]:
res_df.to_csv('model/RNN_record.csv',index=None,encoding='utf-8_sig')

In [ ]:
import matplotlib.pyplot as plt
data = res_df

# 提取唯一的模型名称
models = data['model_name'].unique()

# 绘制验证精度的折线图
plt.figure(figsize=(10, 6))
for model in models:
    model_data = data[data['model_name'] == model]
    plt.plot(model_data['epoch'], model_data['valid_precision'], label=model, marker='o')

# 添加图表标题和标签
plt.title('Validation Precision Across Epochs for Different Models')
plt.xlabel('Epoch')
plt.ylabel('Validation Precision (%)')
plt.legend(title='Model Name')
plt.grid(True)
plt.tight_layout()

# 显示图表
plt.show()

In [ ]:
# 定义所需顺序
desired_order = ['RNN', 'LSTM', 'GRU', 'GRU_bi', 'GRU_bi_3']

# 根据所需顺序重新排序
avg_durations_ordered = avg_durations.loc[desired_order]

# 绘制条形图
fig, ax1 = plt.subplots(figsize=(10, 6))

# 条形宽度
bar_width = 0.4
index = range(len(avg_durations_ordered))

# 左轴：train_duration
ax1.bar(index, avg_durations_ordered['train_duration'], bar_width, label='Train Duration', color='b')
ax1.set_ylabel('Train Duration (s)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_xlabel('Model Name')
ax1.set_xticks(index)
ax1.set_xticklabels(avg_durations_ordered.index, rotation=45)

# 右轴：test_duration
ax2 = ax1.twinx()
ax2.bar([i + bar_width for i in index], avg_durations_ordered['test_duration'], bar_width, label='Test Duration', color='r')
ax2.set_ylabel('Test Duration (s)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

# 添加网格和标题
plt.title('Average Train and Test Durations for Each Model')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 调整布局
fig.tight_layout()

# 显示图表
plt.show()